# Performing flyscan measurements with the controller
awojdyla@lbl.gov, March 2026

In [1]:
from newportxps import NewportXPS

STAGE_NAME = 'Group1.Pos'
IP_ADDRESS = '192.168.10.20'
USERNAME = 'Administrator'
PASSWORD = 'Administrator'

# Connect to the controller (replace with your controller's IP or hostname)
xps = NewportXPS(IP_ADDRESS, username=USERNAME, password=PASSWORD)
# Get a status report
status = xps.status_report()
#print(status)

In [8]:
print(status)

# XPS host:         192.168.10.20 (XPS-0E7C)
# Firmware:         XPS-RL-D-N16001
# Current Time:     Tue Mar 31 10:53:47 2026
# Last Reboot:      Fri Feb 20 19:39:38 2026
# Trajectory Group: None
# Groups and Stages
Group1 (singleaxisinuse), Status: Ready state from motion
# Group1.Pos (XM@XML210@XPS-DRV11)
      Hardware Status: First driver powered on - ZM low level
      Positioner Errors: OK


## attempt at going low-level to get non-blocking 

In [31]:
#This works, but it is still blocking
a = xps._xps.GroupMoveAbsolute(xps._sid, STAGE_NAME, (10,))

## Even lower level
We attempt to directly talk to the socket, and make it non blocking 
#https://github.com/pyepics/newportxps/blob/master/newportxps/XPS_C8_drivers.py


In [ ]:
xps._xps._XPS__sockets[0].setblocking(False)
xps._xps._XPS__sockets[0].getblocking()

In [51]:
# It turns out this is blocking too, because the TCP call sets it as blocking.
xps._xps._XPS__sockets[0].setblocking(False)
xps._xps.GroupMoveAbsolute(xps._sid, STAGE_NAME, (10,))
xps._xps._XPS__sockets[0].getblocking()


(0, '')

In [63]:
xps._xps.TCP_SetTimeout(socketId=0, timeOut=0)
xps._xps.GroupMoveAbsolute(xps._sid, STAGE_NAME, (10,))

(0, '')

In [ ]:
#this automatically get set in send and receive so there is no luck
xps._xps._XPS__sockets[0].timeout
# we could try momey patching but that would be quite difficult.


3600.0

## Using threads
not a great idea but it just works...

In [32]:
from concurrent.futures import ThreadPoolExecutor

# create once, reuse across moves
if "move_executor" not in globals():
    move_executor = ThreadPoolExecutor(max_workers=1)

# start non-blocking move
target = 00.0
move_future = move_executor.submit(
    xps._xps.GroupMoveAbsolute,
    xps._sid,
    STAGE_NAME,
    (target,),
)

print(f"Move to {target} started (non-blocking).")
print("Check status with: move_future.done()")
print("Get result with: move_future.result()")

Move to 0.0 started (non-blocking).
Check status with: move_future.done()
Get result with: move_future.result()


## trying trajectories

see https://github.com/pyepics/newportxps/

https://github.com/pyepics/newportxps/blob/master/newportxps/newportxps.py

```
 # build template for linear trajectory file:
        trajline1 = ['%(ramptime)f']
        trajline2 = ['%(scantime)f']
        trajline3 = ['%(ramptime)f']
        for p in self.traj_positioners:
            trajline1.append('%%(%s_ramp)f' % p)
            trajline1.append('%%(%s_velo)f' % p)
            trajline2.append('%%(%s_dist)f' % p)
            trajline2.append('%%(%s_velo)f' % p)
            trajline3.append('%%(%s_ramp)f' % p)
            trajline3.append('%8.6f' % 0.0)
        trajline1 = (','.join(trajline1)).strip()
        trajline2 = (','.join(trajline2)).strip()
        trajline3 = (','.join(trajline3)).strip()
        self.linear_template = '\n'.join(['', trajline1, trajline2, trajline3])
        self.linear_template = '\n'.join(['', trajline1, trajline2, trajline3, ''])
```

In [18]:
# Use group=None to let the library auto-detect the trajectory group
xps.define_line_trajectories(STAGE_NAME, group=None, pixeltime=0.01, scantime=1.0, start=0, stop=1, step=0.01, accel=None, upload=True, verbose=True)

AttributeError: 'NewportXPS' object has no attribute 'traj_groups'

In [11]:
help(xps.define_line_trajectories)

Help on method define_line_trajectories in module newportxps.newportxps:

define_line_trajectories(*args, **kwargs) method of newportxps.newportxps.NewportXPS instance
    defines 'forward' and 'backward' trajectories for a simple
    single element line scan using PVT Mode



In [ ]:
xps.arm_trajectory(self, name, verbose=False, move_to_start=True, group=None):

```
def run_trajectory(self, name=None, save=True, clean=False, group=None,
                       output_file='Gather.dat', verbose=False, move_to_start=True):

        """run a trajectory in PVT mode

        The trajectory *must be in the ARMED state
        """
```